# BnSentMix — Model 6: Romanization-Consistency Clause-Aware Dual-Polarity XLM-R

## What Model 5 established

Model 5 **rejected H5**. Across seeds 13, 42, and 77, it obtained mean validation macro-F1 `0.7086 ± 0.0055` and mixed-F1 `0.6135 ± 0.0070`. Relative to Model 4, the changes were `−0.0044` macro-F1 and `−0.0123` mixed-F1. Mixed recall remained almost unchanged (`0.5223` versus Model 4's `0.5326`), while precision fell.

The structural analysis is still informative. For reference seed 42, mixed-F1 was only `0.2202` on single-clause examples, but `0.7097` on multi-clause examples and `0.8298` on examples containing an explicit contrast marker. Thus, clause-aware pooling helps when the discourse structure is overt, but the fixed clause heuristic does not improve the complete validation distribution.

All three runs selected epoch 5, and the positive/negative evidence heads remained reasonably strong. The negative result therefore does not look like a failed optimization run. Model 6 keeps the already frozen Model 5 architecture and tests a **separate robustness hypothesis**: whether predictions can be made invariant to plausible Romanized spelling variation.


## Hypothesis H6 and literature basis

**H6:** Relative to a matched clean-only Model 5 control, training on paired clean/perturbed texts with a consistency objective will reduce the clean-to-perturbed macro-F1 drop and prediction-flip rate, while preserving clean macro-F1 and perturbed mixed-F1.

This hypothesis follows the dataset rather than adding an unrelated architecture. The [BnSentMix paper](https://aclanthology.org/2025.loreslm-1.4/) deliberately retains real social-media typing and grammatical errors. Bengali-English transliteration research reports that the same Bengali word is often Romanized differently because users do not follow a single standard ([Mandal and Nanmaran, 2018](https://aclanthology.org/W18-6107/)). More generally, spelling variation is a recognized obstacle in code-mixed NLP ([Yadav et al., 2022](https://aclanthology.org/2022.icon-main.33/)). Consistency training encourages a model to make similar predictions for an input and a meaning-preserving perturbed version ([Xie et al., 2020](https://proceedings.neurips.cc/paper/2020/hash/44feb0096faa8326192570788b38c1d1-Abstract.html)).

Model 6 does **not** introduce a new backbone, class weights, threshold tuning, or another sentiment head. It changes only the training exposure and loss.


## Frozen intervention and decision rules

| Item | Fixed value |
|---|---|
| Parent architecture | Model 5 clause-aware dual-polarity XLM-R |
| Matched control | Same architecture, clean-only BCE, rerun in this notebook |
| Model 6 supervised loss | Mean BCE on clean and perturbed copies |
| Consistency loss | Bernoulli Jensen–Shannon divergence between two evidence outputs |
| Consistency weight | `λ = 1.0` |
| Perturbations | vowel length/omission, consonant or digraph alternatives, repeated characters, and clitic joining |
| Perturbation strength | At most 2 changed Latin tokens plus at most 1 clitic join per text |
| Robust validation | 3 deterministic variants per validation example |
| Backbone / clauses | Identical to Model 5; 8 clauses and length 128 |
| Effective batch | 64 paired examples; micro-batch 8 and accumulation 4 for memory |
| Seeds | 13, 42, 77 |
| Checkpoint selection | Clean validation macro-F1 only |
| Evidence threshold | Fixed at 0.5 |
| Final test | Sealed |

For Model 6,

\[
L=\tfrac{1}{2}\left[\operatorname{BCE}(y,z_c)+\operatorname{BCE}(y,z_p)\right]
+\lambda\operatorname{JS}_{Bernoulli}(\sigma(z_c),\sigma(z_p)).
\]

**H6 is supported only if all four mean criteria hold:**

1. clean macro-F1 difference versus the matched control is at least `−0.005`;
2. macro-F1 robustness-gap reduction is at least `0.005`;
3. prediction-flip-rate reduction is at least `0.01` absolute;
4. perturbed mixed-F1 improves by at least `0.005`.

The earlier central-RQ check is reported separately: clean mixed-F1 must exceed Model 2's `0.6282`, while clean macro-F1 remains at least `0.7132`. These rules are frozen before training.


In [ ]:
# Imports and every experimental choice are declared before training.
from pathlib import Path
from difflib import SequenceMatcher
import gc
import hashlib
import inspect
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from IPython.display import display
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModel, AutoTokenizer, Trainer, TrainerCallback,
    TrainingArguments, set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput

DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model6_romanization_consistency')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026

LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3
TARGET_MAP = {0: [1.0, 0.0], 1: [0.0, 1.0], 2: [0.0, 0.0], 3: [1.0, 1.0]}
EVIDENCE_THRESHOLD = 0.5

MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 128
MAX_CLAUSES = 8
CONTRAST_MARKERS = (
    'but', 'kintu', 'kinto', 'tobe', 'jodio',
    'however', 'although', 'though', 'yet'
)

# Two paired forward passes require a smaller micro-batch. The global effective
# batch and number of optimizer updates remain the same as Models 4 and 5.
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
RUN_SEEDS = [13, 42, 77]
REFERENCE_SEED = 42

# Frozen Romanization-noise protocol.
TOKEN_CHANGE_PROBABILITY = 0.35
MAX_TOKEN_CHANGES = 2
CLITIC_JOIN_PROBABILITY = 0.20
N_VALIDATION_VARIANTS = 3
ROBUSTNESS_SEED = 6006
TRAIN_AUGMENTATION_BASE_SEED = 16000
CONSISTENCY_WEIGHT = 1.0

# Predeclared H6 thresholds.
CLEAN_MACRO_NONINFERIORITY = 0.005
MIN_MACRO_GAP_REDUCTION = 0.005
MIN_FLIP_RATE_REDUCTION = 0.01
MIN_NOISY_MIXED_GAIN = 0.005

MODEL5_MEAN_MACRO_F1 = 0.7086
MODEL5_MEAN_MIXED_F1 = 0.6135
MODEL2_MEAN_MACRO_F1 = 0.7182
MODEL2_MEAN_MIXED_F1 = 0.6282
CENTRAL_MACRO_MARGIN = 0.005

OPEN_FINAL_TEST = False
N_BOOTSTRAP = 2000

USE_FP16 = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('GPU count:', GPU_COUNT)
print('Device 0:', torch.cuda.get_device_name(0) if GPU_COUNT else 'CPU')
print('Consistency weight:', CONSISTENCY_WEIGHT)
print('Robust validation variants:', N_VALIDATION_VARIANTS)
print('Final test open:', OPEN_FINAL_TEST)


In [ ]:
# Reconstruct and verify the identical Models 1–5 split.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, 'Raw dataset differs from Models 1–5.'
lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)
df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id
df['split'] = np.where(fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test'))
assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS

split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
assert sha256_file(split_path) == EXPECTED_SPLIT_SHA256, 'Split differs from Models 1–5.'

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()
display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)


In [ ]:
# The confirmed label mapping and Model 5 dual-polarity targets are unchanged.
def labels_to_evidence(labels):
    return np.asarray([TARGET_MAP[int(label)] for label in labels], dtype=np.float32)


def evidence_to_labels(evidence):
    evidence = np.asarray(evidence).astype(int)
    positive_on = evidence[:, 0] == 1
    negative_on = evidence[:, 1] == 1
    labels = np.full(len(evidence), 2, dtype=int)
    labels[positive_on & ~negative_on] = 0
    labels[~positive_on & negative_on] = 1
    labels[positive_on & negative_on] = 3
    return labels


train_evidence = labels_to_evidence(train_df['label'])
assert np.array_equal(evidence_to_labels(train_evidence), train_df['label'].to_numpy())
display(pd.DataFrame({
    'evidence': ['positive', 'negative'],
    'positive_examples': train_evidence.sum(axis=0).astype(int),
    'training_prevalence': train_evidence.mean(axis=0),
}).round(4))


In [ ]:
# Conservative, deterministic Romanization-like perturbations.
# Labels are never inspected when a variant is generated.
LATIN_WORD_REGEX = re.compile(r'[A-Za-z]+')
CLITIC_REGEX = re.compile(
    r'\b([A-Za-z]{2,})\s+(ta|ti|gula|gulo|er|ke|te)\b',
    flags=re.IGNORECASE,
)
DIGRAPH_ALTERNATIVES = (
    ('sh', 's'), ('ph', 'f'), ('bh', 'b'), ('kh', 'k'),
    ('gh', 'g'), ('ch', 'c'), ('th', 't'), ('dh', 'd'),
)
CONSONANT_ALTERNATIVES = (('z', 'j'), ('j', 'z'), ('v', 'b'), ('b', 'v'))


def stable_uint32(*parts):
    payload = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], 'little')


def restore_case(original, changed_lower):
    if original.isupper():
        return changed_lower.upper()
    if original[:1].isupper() and original[1:].islower():
        return changed_lower.capitalize()
    return changed_lower


def spelling_candidates(token):
    lower = token.lower()
    candidates = {}

    # Common aspiration/digraph variants: bhalo↔balo, kharap↔karap, etc.
    for source, replacement in DIGRAPH_ALTERNATIVES:
        start = lower.find(source)
        if start >= 0:
            changed = lower[:start] + replacement + lower[start + len(source):]
            candidates[changed] = 'digraph_reduction'

    # Common consonant alternatives seen in informal transliteration.
    for source, replacement in CONSONANT_ALTERNATIVES:
        start = lower.find(source)
        if start >= 0:
            changed = lower[:start] + replacement + lower[start + 1:]
            candidates[changed] = 'consonant_alternative'

    # Collapse an existing repeated vowel.
    repeated = re.search(r'([aeiou])\1+', lower)
    if repeated:
        start, end = repeated.span()
        changed = lower[:start] + repeated.group(1) + lower[end:]
        candidates[changed] = 'vowel_collapse'

    internal_vowels = [
        index for index, char in enumerate(lower)
        if char in 'aeiou' and 0 < index < len(lower) - 1
    ]
    if internal_vowels:
        index = internal_vowels[0]
        candidates[lower[:index] + lower[index] + lower[index:]] = 'vowel_repeat'
        if len(lower) >= 5:
            candidates[lower[:index] + lower[index + 1:]] = 'vowel_omission'

    # Short tokens often have no internal vowel; repeat one existing vowel.
    if not candidates:
        vowel_positions = [i for i, char in enumerate(lower) if char in 'aeiou']
        if vowel_positions:
            index = vowel_positions[0]
            candidates[lower[:index] + lower[index] + lower[index:]] = 'vowel_repeat'

    candidates.pop(lower, None)
    return [
        (restore_case(token, changed), operation)
        for changed, operation in sorted(candidates.items())
    ]


def romanization_variant(text, random_seed):
    text = str(text)
    rng = np.random.default_rng(random_seed)
    matches = list(LATIN_WORD_REGEX.finditer(text))
    eligible = []
    for match in matches:
        token = match.group(0)
        if len(token) >= 3:
            candidates = spelling_candidates(token)
            if candidates:
                eligible.append((match, candidates))

    selected = []
    if eligible:
        order = rng.permutation(len(eligible))
        for position in order:
            if rng.random() < TOKEN_CHANGE_PROBABILITY:
                selected.append(eligible[int(position)])
            if len(selected) == MAX_TOKEN_CHANGES:
                break
        if not selected:
            selected = [eligible[int(order[0])]]

    replacements = {}
    operations = []
    for match, candidates in selected:
        replacement, operation = candidates[int(rng.integers(len(candidates)))]
        replacements[(match.start(), match.end())] = replacement
        operations.append(operation)

    pieces, cursor = [], 0
    for match in matches:
        pieces.append(text[cursor:match.start()])
        pieces.append(replacements.get((match.start(), match.end()), match.group(0)))
        cursor = match.end()
    pieces.append(text[cursor:])
    variant = ''.join(pieces)

    # Joining an explicit Bangla clitic models spacing/affix variation while
    # preserving every character and the intended meaning.
    clitic_matches = list(CLITIC_REGEX.finditer(variant))
    if clitic_matches and rng.random() < CLITIC_JOIN_PROBABILITY:
        match = clitic_matches[int(rng.integers(len(clitic_matches)))]
        variant = variant[:match.start()] + match.group(1) + match.group(2) + variant[match.end():]
        operations.append('clitic_join')

    return variant, operations


def build_variant_frame(frame, base_seed, variant_id):
    rows = []
    for row in frame[['source_row_id', 'group_id', 'text', 'label']].itertuples(index=False):
        row_seed = stable_uint32(base_seed, row.source_row_id, variant_id)
        variant_text, operations = romanization_variant(row.text, row_seed)
        rows.append({
            'source_row_id': row.source_row_id,
            'group_id': row.group_id,
            'label': row.label,
            'original_text': row.text,
            'variant_text': variant_text,
            'operations': '|'.join(operations) if operations else 'unchanged',
            'changed': variant_text != row.text,
            'character_change_ratio': 1.0 - SequenceMatcher(
                None, row.text, variant_text
            ).ratio(),
        })
    return pd.DataFrame(rows)


# Audit only training and validation text; the test partition remains untouched.
train_augmentation_audit = build_variant_frame(train_df, ROBUSTNESS_SEED, 0)
validation_variants = [
    build_variant_frame(val_df, ROBUSTNESS_SEED, variant_id)
    for variant_id in range(N_VALIDATION_VARIANTS)
]
audit_summary = train_augmentation_audit.groupby('label').agg(
    samples=('source_row_id', 'size'),
    changed_rate=('changed', 'mean'),
    mean_character_change=('character_change_ratio', 'mean'),
).rename(index=LABEL_NAMES)
display(audit_summary.round(4))
display(train_augmentation_audit[
    train_augmentation_audit['changed']
].sample(8, random_state=ROBUSTNESS_SEED)[[
    'original_text', 'variant_text', 'operations'
]])

# This is a perturbation-validity audit, not another label audit.
audit_sample = validation_variants[0].sample(
    min(300, len(validation_variants[0])), random_state=ROBUSTNESS_SEED
)
audit_sample.to_csv(MODEL_DIR / 'manual_perturbation_audit_300.csv', index=False)


In [ ]:
# Clause construction is copied unchanged from Model 5.
CONTRAST_REGEX = re.compile(
    r'\b(?:' + '|'.join(map(re.escape, CONTRAST_MARKERS)) + r')\b',
    flags=re.IGNORECASE,
)
PUNCTUATION_REGEX = re.compile(r'[.!?,;]+')


def split_clauses(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if not text:
        return ['']
    boundaries = set()
    for match in PUNCTUATION_REGEX.finditer(text):
        if 0 < match.end() < len(text):
            boundaries.add(match.end())
    for match in CONTRAST_REGEX.finditer(text):
        if 0 < match.start() < len(text):
            boundaries.add(match.start())

    clauses, start = [], 0
    for boundary in sorted(boundaries) + [len(text)]:
        clause = text[start:boundary].strip()
        if clause:
            clauses.append(clause)
        start = boundary
    if not clauses:
        clauses = [text]
    if len(clauses) > MAX_CLAUSES:
        clauses = clauses[:MAX_CLAUSES - 1] + [' '.join(clauses[MAX_CLAUSES - 1:])]
    return clauses


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def balanced_prefix_lengths(tokenized_clauses, budget):
    allocations = [0] * len(tokenized_clauses)
    while budget > 0:
        progressed = False
        for index, token_ids in enumerate(tokenized_clauses):
            if budget == 0:
                break
            if allocations[index] < len(token_ids):
                allocations[index] += 1
                budget -= 1
                progressed = True
        if not progressed:
            break
    return allocations


def encode_clause_text(text):
    clauses = split_clauses(text)
    tokenized_clauses = [
        tokenizer.encode(clause, add_special_tokens=False) for clause in clauses
    ]
    separator_count = max(len(clauses) - 1, 0)
    token_budget = MAX_LENGTH - 2 - separator_count
    allocations = balanced_prefix_lengths(tokenized_clauses, token_budget)

    input_ids = [tokenizer.cls_token_id]
    clause_ids = [-1]
    for clause_index, (token_ids, keep_count) in enumerate(
        zip(tokenized_clauses, allocations)
    ):
        if clause_index > 0:
            input_ids.append(tokenizer.sep_token_id)
            clause_ids.append(-1)
        input_ids.extend(token_ids[:keep_count])
        clause_ids.extend([clause_index] * keep_count)
    input_ids.append(tokenizer.sep_token_id)
    clause_ids.append(-1)
    assert len(input_ids) <= MAX_LENGTH
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'clause_ids': clause_ids,
    }


class PairedClauseDataset(Dataset):
    def __init__(self, frame, augmentation_seed, variant_id=0):
        self.clean_texts = frame['text'].tolist()
        variants = build_variant_frame(frame, augmentation_seed, variant_id)
        self.augmented_texts = variants['variant_text'].tolist()
        self.targets = labels_to_evidence(frame['label']).tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        clean = encode_clause_text(self.clean_texts[index])
        augmented = encode_clause_text(self.augmented_texts[index])
        item = dict(clean)
        item.update({f'aug_{key}': value for key, value in augmented.items()})
        item['labels'] = self.targets[index]
        return item


class ClauseInferenceDataset(Dataset):
    def __init__(self, texts):
        self.texts = list(texts)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return encode_clause_text(self.texts[index])


def pad_clause_features(features):
    features = [dict(feature) for feature in features]
    clause_ids = [feature.pop('clause_ids') for feature in features]
    batch = tokenizer.pad(
        features, padding=True,
        pad_to_multiple_of=8 if USE_FP16 else None,
        return_tensors='pt',
    )
    padded_length = batch['input_ids'].shape[1]
    batch['clause_ids'] = torch.tensor([
        ids + [-1] * (padded_length - len(ids)) for ids in clause_ids
    ], dtype=torch.long)
    return batch


class PairedClauseCollator:
    def __call__(self, features):
        features = [dict(feature) for feature in features]
        labels = [feature.pop('labels') for feature in features]
        clean_features, augmented_features = [], []
        for feature in features:
            clean_features.append({
                key: value for key, value in feature.items()
                if not key.startswith('aug_')
            })
            augmented_features.append({
                key.removeprefix('aug_'): value
                for key, value in feature.items() if key.startswith('aug_')
            })
        batch = pad_clause_features(clean_features)
        augmented_batch = pad_clause_features(augmented_features)
        batch.update({f'aug_{key}': value for key, value in augmented_batch.items()})
        batch['labels'] = torch.tensor(labels, dtype=torch.float32)
        return batch


class ClauseInferenceCollator:
    def __call__(self, features):
        return pad_clause_features(features)


paired_collator = PairedClauseCollator()
inference_collator = ClauseInferenceCollator()

VISIBLE_DEVICES = max(GPU_COUNT, 1)
GLOBAL_EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE * VISIBLE_DEVICES * GRADIENT_ACCUMULATION_STEPS
)
BATCHES_PER_EPOCH = math.ceil(len(train_df) / (TRAIN_BATCH_SIZE * VISIBLE_DEVICES))
UPDATES_PER_EPOCH = math.ceil(BATCHES_PER_EPOCH / GRADIENT_ACCUMULATION_STEPS)
TOTAL_UPDATE_STEPS = UPDATES_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = math.ceil(TOTAL_UPDATE_STEPS * WARMUP_FRACTION)
print('Training / validation:', len(train_df), '/', len(val_df))
print('Global effective paired batch:', GLOBAL_EFFECTIVE_BATCH_SIZE)
print('Maximum optimizer steps / warmup:', TOTAL_UPDATE_STEPS, '/', WARMUP_STEPS)


In [ ]:
# Metrics are identical to Models 4 and 5.
def sigmoid_numpy(logits):
    logits = np.clip(np.asarray(logits), -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))


def predictions_from_logits(logits):
    probabilities = sigmoid_numpy(logits)
    evidence_predictions = (probabilities >= EVIDENCE_THRESHOLD).astype(int)
    return probabilities, evidence_predictions, evidence_to_labels(evidence_predictions)


def calculate_metrics(true_evidence, predicted_evidence):
    true_evidence = np.asarray(true_evidence).astype(int)
    predicted_evidence = np.asarray(predicted_evidence).astype(int)
    labels = evidence_to_labels(true_evidence)
    predictions = evidence_to_labels(predicted_evidence)
    mixed_precision, mixed_recall, mixed_f1, _ = precision_recall_fscore_support(
        labels == MIXED_LABEL, predictions == MIXED_LABEL,
        average='binary', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='macro', zero_division=0
        )),
        'weighted_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='weighted', zero_division=0
        )),
        'mixed_precision': float(mixed_precision),
        'mixed_recall': float(mixed_recall),
        'mixed_f1': float(mixed_f1),
        'positive_evidence_f1': float(f1_score(
            true_evidence[:, 0], predicted_evidence[:, 0], zero_division=0
        )),
        'negative_evidence_f1': float(f1_score(
            true_evidence[:, 1], predicted_evidence[:, 1], zero_division=0
        )),
        'evidence_exact_match': float(np.mean(np.all(
            true_evidence == predicted_evidence, axis=1
        ))),
    }


def compute_clean_metrics(eval_prediction):
    logits, true_evidence = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    _, predicted_evidence, _ = predictions_from_logits(logits)
    return calculate_metrics(true_evidence, predicted_evidence)


In [ ]:
# Architecture is exactly Model 5: separate polarity queries over clause means.
class ClauseAwareDualPolarityModel(nn.Module):
    def __init__(self, model_name, max_clauses):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        self.config.num_labels = 2
        self.config.problem_type = 'multi_label_classification'
        self.max_clauses = int(max_clauses)
        hidden_size = self.config.hidden_size
        dropout_rate = getattr(self.config, 'classifier_dropout', None)
        if dropout_rate is None:
            dropout_rate = self.config.hidden_dropout_prob
        self.dropout = nn.Dropout(dropout_rate)
        self.positive_query = nn.Parameter(torch.empty(hidden_size))
        self.negative_query = nn.Parameter(torch.empty(hidden_size))
        self.positive_head = nn.Linear(hidden_size * 2, 1)
        self.negative_head = nn.Linear(hidden_size * 2, 1)
        self._initialize_new_parameters()

    def _initialize_new_parameters(self):
        std = getattr(self.config, 'initializer_range', 0.02)
        nn.init.normal_(self.positive_query, mean=0.0, std=std)
        nn.init.normal_(self.negative_query, mean=0.0, std=std)
        for head in (self.positive_head, self.negative_head):
            nn.init.normal_(head.weight, mean=0.0, std=std)
            nn.init.zeros_(head.bias)

    @staticmethod
    def _query_pool(clause_vectors, valid_clauses, query):
        scores = torch.einsum('bkh,h->bk', clause_vectors, query)
        scores = scores / math.sqrt(clause_vectors.shape[-1])
        scores = scores.masked_fill(~valid_clauses, -1e4)
        weights = torch.softmax(scores, dim=1) * valid_clauses.to(scores.dtype)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        return torch.sum(clause_vectors * weights.unsqueeze(-1), dim=1)

    def forward(
        self, input_ids=None, attention_mask=None, clause_ids=None,
        labels=None, **kwargs
    ):
        encoder_output = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask, **kwargs
        )
        token_vectors = encoder_output.last_hidden_state
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        clause_vectors, valid_clauses = [], []
        for clause_index in range(self.max_clauses):
            mask = (clause_ids == clause_index) & attention_mask.bool()
            counts = mask.sum(dim=1, keepdim=True)
            pooled = torch.sum(token_vectors * mask.unsqueeze(-1), dim=1)
            pooled = pooled / counts.clamp_min(1).to(token_vectors.dtype)
            clause_vectors.append(pooled)
            valid_clauses.append(counts.squeeze(1) > 0)
        clause_vectors = torch.stack(clause_vectors, dim=1)
        valid_clauses = torch.stack(valid_clauses, dim=1)

        positive_context = self._query_pool(
            clause_vectors, valid_clauses, self.positive_query
        )
        negative_context = self._query_pool(
            clause_vectors, valid_clauses, self.negative_query
        )
        global_context = token_vectors[:, 0, :]
        positive_logit = self.positive_head(self.dropout(torch.cat(
            [global_context, positive_context], dim=1
        )))
        negative_logit = self.negative_head(self.dropout(torch.cat(
            [global_context, negative_context], dim=1
        )))
        logits = torch.cat([positive_logit, negative_logit], dim=1)
        return SequenceClassifierOutput(logits=logits)


preview_model = ClauseAwareDualPolarityModel(MODEL_NAME, MAX_CLAUSES)
print(f'Total parameters: {sum(p.numel() for p in preview_model.parameters()):,}')
del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Matched control uses only clean BCE. Model 6 adds augmented BCE and JS consistency.
def bernoulli_js_divergence(clean_logits, augmented_logits, epsilon=1e-6):
    clean_p = torch.sigmoid(clean_logits).clamp(epsilon, 1.0 - epsilon)
    augmented_p = torch.sigmoid(augmented_logits).clamp(epsilon, 1.0 - epsilon)
    midpoint = 0.5 * (clean_p + augmented_p)

    def bernoulli_kl(p, q):
        return p * (torch.log(p) - torch.log(q)) + (
            (1.0 - p) * (torch.log1p(-p) - torch.log1p(-q))
        )

    return 0.5 * (
        bernoulli_kl(clean_p, midpoint) +
        bernoulli_kl(augmented_p, midpoint)
    ).mean()


class RomanizationConsistencyTrainer(Trainer):
    def __init__(
        self, *args, use_augmented_pair=False,
        consistency_weight=0.0, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.use_augmented_pair = bool(use_augmented_pair)
        self.consistency_weight = float(consistency_weight)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        targets = inputs.pop('labels').float()
        augmented_inputs = {
            key.removeprefix('aug_'): inputs.pop(key)
            for key in list(inputs) if key.startswith('aug_')
        }
        clean_outputs = model(**inputs)
        clean_loss = F.binary_cross_entropy_with_logits(
            clean_outputs.logits, targets
        )

        if not self.use_augmented_pair:
            loss = clean_loss
        else:
            augmented_outputs = model(**augmented_inputs)
            augmented_loss = F.binary_cross_entropy_with_logits(
                augmented_outputs.logits, targets
            )
            consistency_loss = bernoulli_js_divergence(
                clean_outputs.logits, augmented_outputs.logits
            )
            loss = 0.5 * (clean_loss + augmented_loss) + (
                self.consistency_weight * consistency_loss
            )
        return (loss, clean_outputs) if return_outputs else loss


class ExactBestStateCallback(TrainerCallback):
    def __init__(self, state_path, patience):
        self.state_path = Path(state_path)
        self.patience = int(patience)
        self.best_score = -np.inf
        self.best_epoch = None
        self.best_loss = None
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get('eval_macro_f1') if metrics else None
        if score is None:
            return control
        if score > self.best_score:
            self.best_score = float(score)
            self.best_epoch = float(state.epoch)
            self.best_loss = float(metrics['eval_loss'])
            self.bad_epochs = 0
            unwrapped_model = model.module if hasattr(model, 'module') else model
            cpu_state = {
                key: value.detach().cpu()
                for key, value in unwrapped_model.state_dict().items()
            }
            torch.save({'model_state_dict': cpu_state}, self.state_path)
        else:
            self.bad_epochs += 1
            if self.bad_epochs >= self.patience:
                control.should_training_stop = True
        return control


def load_exact_state(path):
    load_kwargs = {'map_location': 'cpu'}
    if 'weights_only' in inspect.signature(torch.load).parameters:
        load_kwargs['weights_only'] = True
    return torch.load(path, **load_kwargs)['model_state_dict']


In [ ]:
# Both arms use the same optimizer budget and validation checkpoint rule.
ARM_CONFIGS = {
    'model5_clean_control': {
        'use_augmented_pair': False,
        'consistency_weight': 0.0,
    },
    'model6_consistency': {
        'use_augmented_pair': True,
        'consistency_weight': CONSISTENCY_WEIGHT,
    },
}


def make_training_arguments(seed, output_dir):
    arguments = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='linear',
        max_grad_norm=1.0,
        save_strategy='no',
        logging_strategy='epoch',
        load_best_model_at_end=False,
        seed=seed,
        data_seed=seed,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        report_to=[],
    )
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    arguments[
        'eval_strategy' if 'eval_strategy' in argument_names
        else 'evaluation_strategy'
    ] = 'epoch'
    return TrainingArguments(**arguments)


def make_inference_arguments(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        report_to=[],
    )


def build_fresh_model():
    return ClauseAwareDualPolarityModel(MODEL_NAME, MAX_CLAUSES)


In [ ]:
# Train the matched control and Model 6 for every seed, then evaluate the same
# three frozen perturbation variants. Expect roughly three times Model 5's runtime.
def predict_texts(model, texts, output_dir):
    inference_trainer = Trainer(
        model=model,
        args=make_inference_arguments(output_dir),
        data_collator=inference_collator,
    )
    output = inference_trainer.predict(ClauseInferenceDataset(texts))
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    probabilities, evidence_predictions, class_predictions = predictions_from_logits(logits)
    del inference_trainer, output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return probabilities, evidence_predictions, class_predictions


def evaluate_clean_and_variants(
    model, frame, variants, output_dir, split_name='validation'
):
    true_evidence = labels_to_evidence(frame['label'])
    clean_probabilities, clean_evidence, clean_predictions = predict_texts(
        model, frame['text'].tolist(), output_dir / 'clean_inference'
    )
    clean_metrics = calculate_metrics(true_evidence, clean_evidence)

    variant_rows = []
    variant_predictions = []
    variant_evidence_predictions = []
    variant_probabilities = []
    for variant_id, variant_frame in enumerate(variants):
        probabilities, evidence_predictions, predictions = predict_texts(
            model, variant_frame['variant_text'].tolist(),
            output_dir / f'variant_{variant_id}_inference'
        )
        metrics = calculate_metrics(true_evidence, evidence_predictions)
        changed_mask = variant_frame['changed'].to_numpy(dtype=bool)
        changed_metrics = calculate_metrics(
            true_evidence[changed_mask], evidence_predictions[changed_mask]
        )
        row = {
            'variant_id': variant_id,
            **metrics,
            'changed_support': int(changed_mask.sum()),
            'changed_macro_f1': changed_metrics['macro_f1'],
            'changed_mixed_f1': changed_metrics['mixed_f1'],
            'prediction_flip_rate': float(np.mean(predictions != clean_predictions)),
            'evidence_flip_rate': float(np.mean(np.any(
                evidence_predictions != clean_evidence, axis=1
            ))),
            'mean_probability_shift': float(np.mean(np.abs(
                probabilities - clean_probabilities
            ))),
        }
        variant_rows.append(row)
        variant_predictions.append(predictions)
        variant_evidence_predictions.append(evidence_predictions)
        variant_probabilities.append(probabilities)

        prediction_frame = variant_frame.copy()
        prediction_frame['clean_prediction'] = clean_predictions
        prediction_frame['variant_prediction'] = predictions
        prediction_frame['clean_positive_probability'] = clean_probabilities[:, 0]
        prediction_frame['clean_negative_probability'] = clean_probabilities[:, 1]
        prediction_frame['variant_positive_probability'] = probabilities[:, 0]
        prediction_frame['variant_negative_probability'] = probabilities[:, 1]
        prediction_frame.to_csv(
            output_dir / f'{split_name}_variant_{variant_id}_predictions.csv',
            index=False,
        )

    variant_table = pd.DataFrame(variant_rows)
    summary = {f'clean_{key}': value for key, value in clean_metrics.items()}
    for metric in [
        'accuracy', 'macro_f1', 'weighted_f1', 'mixed_precision',
        'mixed_recall', 'mixed_f1', 'positive_evidence_f1',
        'negative_evidence_f1', 'evidence_exact_match',
        'changed_macro_f1', 'changed_mixed_f1', 'prediction_flip_rate',
        'evidence_flip_rate', 'mean_probability_shift'
    ]:
        summary[f'noisy_{metric}'] = float(variant_table[metric].mean())
    summary['macro_f1_drop'] = summary['clean_macro_f1'] - summary['noisy_macro_f1']
    summary['mixed_f1_drop'] = summary['clean_mixed_f1'] - summary['noisy_mixed_f1']

    bundle = {
        'clean_probabilities': clean_probabilities,
        'clean_evidence_predictions': clean_evidence,
        'clean_predictions': clean_predictions,
        'variant_probabilities': variant_probabilities,
        'variant_evidence_predictions': variant_evidence_predictions,
        'variant_predictions': variant_predictions,
    }
    variant_table.to_csv(output_dir / 'robustness_by_variant.csv', index=False)
    return summary, bundle


def train_one_arm_seed(arm_name, arm_config, seed):
    seed_dir = MODEL_DIR / arm_name / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    best_state_path = seed_dir / 'best_state.pt'
    set_seed(seed)

    train_dataset = PairedClauseDataset(
        train_df, TRAIN_AUGMENTATION_BASE_SEED + seed
    )
    validation_pair_dataset = PairedClauseDataset(
        val_df, ROBUSTNESS_SEED, variant_id=0
    )
    model = build_fresh_model()
    callback = ExactBestStateCallback(best_state_path, EARLY_STOPPING_PATIENCE)
    trainer = RomanizationConsistencyTrainer(
        model=model,
        args=make_training_arguments(seed, seed_dir),
        train_dataset=train_dataset,
        eval_dataset=validation_pair_dataset,
        data_collator=paired_collator,
        compute_metrics=compute_clean_metrics,
        callbacks=[callback],
        **arm_config,
    )
    trainer.train()

    if not best_state_path.exists():
        raise RuntimeError(f'No validation-selected state for {arm_name}, seed {seed}.')
    model.load_state_dict(load_exact_state(best_state_path), strict=True)
    robust_metrics, prediction_bundle = evaluate_clean_and_variants(
        model, val_df, validation_variants, seed_dir
    )
    robust_metrics.update({
        'arm': arm_name,
        'seed': seed,
        'best_epoch': callback.best_epoch,
        'validation_loss': callback.best_loss,
        'best_state_path': str(best_state_path),
    })
    pd.DataFrame(trainer.state.log_history).to_csv(
        seed_dir / 'training_log.csv', index=False
    )

    del trainer, model, train_dataset, validation_pair_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return robust_metrics, prediction_bundle


validation_rows = []
validation_outputs = {}
for arm_name, arm_config in ARM_CONFIGS.items():
    for run_seed in RUN_SEEDS:
        print(f'\n===== {arm_name} | seed {run_seed} =====')
        metrics, output_bundle = train_one_arm_seed(
            arm_name, arm_config, run_seed
        )
        validation_rows.append(metrics)
        validation_outputs[(arm_name, run_seed)] = output_bundle

validation_results = pd.DataFrame(validation_rows).sort_values(
    ['arm', 'seed']
).reset_index(drop=True)
summary_metrics = [
    'clean_accuracy', 'clean_macro_f1', 'clean_weighted_f1',
    'clean_mixed_precision', 'clean_mixed_recall', 'clean_mixed_f1',
    'noisy_macro_f1', 'noisy_mixed_f1', 'macro_f1_drop', 'mixed_f1_drop',
    'noisy_prediction_flip_rate', 'noisy_evidence_flip_rate',
    'noisy_mean_probability_shift', 'noisy_changed_macro_f1',
    'noisy_changed_mixed_f1'
]
validation_summary = validation_results.groupby('arm')[summary_metrics].agg(
    ['mean', 'std']
)
validation_results.to_csv(MODEL_DIR / 'validation_seed_results.csv', index=False)
validation_summary.to_csv(MODEL_DIR / 'validation_mean_std.csv')
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')
display(validation_results.drop(columns='best_state_path').round(4))
display(validation_summary.round(4))


In [ ]:
# Same-seed effects and all predeclared decisions.
control = validation_results[
    validation_results['arm'].eq('model5_clean_control')
].copy()
model6 = validation_results[
    validation_results['arm'].eq('model6_consistency')
].copy()
paired = control.merge(model6, on='seed', suffixes=('_control', '_model6'))
paired['delta_clean_macro'] = (
    paired['clean_macro_f1_model6'] - paired['clean_macro_f1_control']
)
paired['macro_gap_reduction'] = (
    paired['macro_f1_drop_control'] - paired['macro_f1_drop_model6']
)
paired['flip_rate_reduction'] = (
    paired['noisy_prediction_flip_rate_control'] -
    paired['noisy_prediction_flip_rate_model6']
)
paired['delta_noisy_mixed'] = (
    paired['noisy_mixed_f1_model6'] - paired['noisy_mixed_f1_control']
)
paired['delta_noisy_macro'] = (
    paired['noisy_macro_f1_model6'] - paired['noisy_macro_f1_control']
)

comparison_columns = [
    'seed', 'clean_macro_f1_control', 'clean_macro_f1_model6',
    'delta_clean_macro', 'noisy_macro_f1_control', 'noisy_macro_f1_model6',
    'delta_noisy_macro', 'macro_gap_reduction',
    'noisy_prediction_flip_rate_control',
    'noisy_prediction_flip_rate_model6', 'flip_rate_reduction',
    'noisy_mixed_f1_control', 'noisy_mixed_f1_model6',
    'delta_noisy_mixed'
]
display(paired[comparison_columns].round(4))
paired.to_csv(MODEL_DIR / 'paired_model5_control_vs_model6.csv', index=False)

mean_effects = paired[[
    'delta_clean_macro', 'macro_gap_reduction',
    'flip_rate_reduction', 'delta_noisy_mixed', 'delta_noisy_macro'
]].mean()
h6_supported = (
    mean_effects['delta_clean_macro'] >= -CLEAN_MACRO_NONINFERIORITY and
    mean_effects['macro_gap_reduction'] >= MIN_MACRO_GAP_REDUCTION and
    mean_effects['flip_rate_reduction'] >= MIN_FLIP_RATE_REDUCTION and
    mean_effects['delta_noisy_mixed'] >= MIN_NOISY_MIXED_GAIN
)

model6_mean_clean_macro = float(model6['clean_macro_f1'].mean())
model6_mean_clean_mixed = float(model6['clean_mixed_f1'].mean())
historical_model5_noninferior = (
    model6_mean_clean_macro >= MODEL5_MEAN_MACRO_F1 - CLEAN_MACRO_NONINFERIORITY and
    model6_mean_clean_mixed >= MODEL5_MEAN_MIXED_F1 - CLEAN_MACRO_NONINFERIORITY
)
central_rq_supported = (
    model6_mean_clean_mixed > MODEL2_MEAN_MIXED_F1 and
    model6_mean_clean_macro >= MODEL2_MEAN_MACRO_F1 - CENTRAL_MACRO_MARGIN
)
control_reproduction_within_margin = (
    abs(float(control['clean_macro_f1'].mean()) - MODEL5_MEAN_MACRO_F1) <= 0.005 and
    abs(float(control['clean_mixed_f1'].mean()) - MODEL5_MEAN_MIXED_F1) <= 0.005
)

decision = pd.DataFrame([{
    **mean_effects.to_dict(),
    'model6_mean_clean_macro_f1': model6_mean_clean_macro,
    'model6_mean_clean_mixed_f1': model6_mean_clean_mixed,
    'matched_control_reproduces_model5': control_reproduction_within_margin,
    'H6_supported': h6_supported,
    'clean_noninferior_to_historical_model5': historical_model5_noninferior,
    'central_RQ_supported_vs_Model2': central_rq_supported,
}])
decision.to_csv(MODEL_DIR / 'hypothesis_H6_decision.csv', index=False)
display(decision.round(4))
print('H6 robustness decision:', h6_supported)
print('Central RQ decision versus Model 2:', central_rq_supported)


In [ ]:
# Reference-seed error analysis: clean/noisy reports and perturbation families.
true_labels = val_df['label'].to_numpy()
reference_control = validation_outputs[('model5_clean_control', REFERENCE_SEED)]
reference_model6 = validation_outputs[('model6_consistency', REFERENCE_SEED)]

for name, bundle in [
    ('Matched Model 5 control', reference_control),
    ('Model 6 consistency', reference_model6),
]:
    print(f'\n{name} — clean validation')
    display(pd.DataFrame(classification_report(
        true_labels, bundle['clean_predictions'], labels=LABELS,
        target_names=[LABEL_NAMES[label] for label in LABELS],
        output_dict=True, zero_division=0
    )).T.round(4))
    print(f'{name} — perturbation variant 0 confusion matrix')
    display(pd.DataFrame(
        confusion_matrix(
            true_labels, bundle['variant_predictions'][0], labels=LABELS
        ),
        index=[f'true_{LABEL_NAMES[label]}' for label in LABELS],
        columns=[f'pred_{LABEL_NAMES[label]}' for label in LABELS],
    ))

operation_families = {
    'vowel': 'vowel_',
    'digraph': 'digraph_',
    'consonant': 'consonant_',
    'clitic_join': 'clitic_join',
}
operation_rows = []
for variant_id, variant_frame in enumerate(validation_variants):
    for operation_name, pattern in operation_families.items():
        mask = variant_frame['operations'].str.contains(
            pattern, regex=False
        ).to_numpy()
        if not mask.any():
            continue
        for arm_name, bundle in [
            ('model5_clean_control', reference_control),
            ('model6_consistency', reference_model6),
        ]:
            flips = (
                bundle['variant_predictions'][variant_id] !=
                bundle['clean_predictions']
            )
            operation_rows.append({
                'variant_id': variant_id,
                'operation': operation_name,
                'arm': arm_name,
                'support': int(mask.sum()),
                'prediction_flip_rate': float(flips[mask].mean()),
            })
operation_table = pd.DataFrame(operation_rows)
operation_summary = operation_table.groupby(
    ['operation', 'arm'], as_index=False
).agg(
    mean_support=('support', 'mean'),
    mean_flip_rate=('prediction_flip_rate', 'mean'),
)
operation_summary.to_csv(
    MODEL_DIR / 'reference_seed_robustness_by_operation.csv', index=False
)
display(operation_summary.round(4))

# Does consistency preserve simultaneous evidence activation for true mixed text?
mixed_mask = true_labels == MIXED_LABEL
coactivation_rows = []
for arm_name, bundle in [
    ('model5_clean_control', reference_control),
    ('model6_consistency', reference_model6),
]:
    clean_both = np.all(
        bundle['clean_evidence_predictions'][mixed_mask] == 1, axis=1
    ).mean()
    noisy_both = np.mean([
        np.all(evidence[mixed_mask] == 1, axis=1).mean()
        for evidence in bundle['variant_evidence_predictions']
    ])
    coactivation_rows.append({
        'arm': arm_name,
        'clean_mixed_both_heads_on': clean_both,
        'noisy_mixed_both_heads_on': noisy_both,
        'coactivation_drop': clean_both - noisy_both,
    })
display(pd.DataFrame(coactivation_rows).round(4))


In [ ]:
# Clustered paired bootstrap for the reference seed.
# Original examples are resampled; all three variants of a sampled example stay together.
def noisy_metric_mean(labels, variant_predictions, metric_name, indices):
    values = []
    for predictions in variant_predictions:
        if metric_name == 'macro_f1':
            value = f1_score(
                labels[indices], predictions[indices], labels=LABELS,
                average='macro', zero_division=0
            )
        elif metric_name == 'mixed_f1':
            value = f1_score(
                labels[indices] == MIXED_LABEL,
                predictions[indices] == MIXED_LABEL,
                average='binary', zero_division=0
            )
        else:
            raise ValueError(metric_name)
        values.append(value)
    return float(np.mean(values))


def paired_bootstrap_delta(metric_name, n_bootstrap=N_BOOTSTRAP):
    rng = np.random.default_rng(ROBUSTNESS_SEED + REFERENCE_SEED)
    n = len(true_labels)
    deltas = np.empty(n_bootstrap, dtype=np.float32)
    for index in range(n_bootstrap):
        sample = rng.integers(0, n, size=n)
        control_value = noisy_metric_mean(
            true_labels, reference_control['variant_predictions'],
            metric_name, sample
        )
        model6_value = noisy_metric_mean(
            true_labels, reference_model6['variant_predictions'],
            metric_name, sample
        )
        deltas[index] = model6_value - control_value
    return {
        'metric': metric_name,
        'model6_minus_control': float(deltas.mean()),
        'ci_2.5': float(np.quantile(deltas, 0.025)),
        'ci_97.5': float(np.quantile(deltas, 0.975)),
        'bootstrap_samples': n_bootstrap,
    }


bootstrap_table = pd.DataFrame([
    paired_bootstrap_delta('macro_f1'),
    paired_bootstrap_delta('mixed_f1'),
])
bootstrap_table.to_csv(
    MODEL_DIR / 'reference_seed_paired_bootstrap.csv', index=False
)
display(bootstrap_table.round(4))


In [ ]:
# Save the complete frozen experiment specification.
manifest = {
    'experiment': 'Model 6: Romanization-consistency clause-aware dual-polarity XLM-R',
    'hypothesis': 'Consistency training reduces Romanization-noise sensitivity',
    'dataset_sha256': EXPECTED_RAW_SHA256,
    'split_sha256': EXPECTED_SPLIT_SHA256,
    'model_name': MODEL_NAME,
    'parent_architecture': 'Model 5 clause-aware dual-polarity XLM-R',
    'matched_control': 'clean-only Model 5 architecture rerun in this notebook',
    'target_map': TARGET_MAP,
    'contrast_markers': list(CONTRAST_MARKERS),
    'max_clauses': MAX_CLAUSES,
    'max_length': MAX_LENGTH,
    'evidence_threshold': EVIDENCE_THRESHOLD,
    'romanization_noise': {
        'token_change_probability': TOKEN_CHANGE_PROBABILITY,
        'max_token_changes': MAX_TOKEN_CHANGES,
        'clitic_join_probability': CLITIC_JOIN_PROBABILITY,
        'validation_variants': N_VALIDATION_VARIANTS,
        'robustness_seed': ROBUSTNESS_SEED,
    },
    'consistency_loss': 'Bernoulli Jensen-Shannon divergence',
    'consistency_weight': CONSISTENCY_WEIGHT,
    'run_seeds': RUN_SEEDS,
    'global_effective_batch_size': GLOBAL_EFFECTIVE_BATCH_SIZE,
    'h6_thresholds': {
        'clean_macro_noninferiority': CLEAN_MACRO_NONINFERIORITY,
        'minimum_macro_gap_reduction': MIN_MACRO_GAP_REDUCTION,
        'minimum_flip_rate_reduction': MIN_FLIP_RATE_REDUCTION,
        'minimum_noisy_mixed_gain': MIN_NOISY_MIXED_GAIN,
    },
    'test_opened': OPEN_FINAL_TEST,
}
with open(MODEL_DIR / 'experiment_manifest.json', 'w') as handle:
    json.dump(manifest, handle, indent=2)


## Final-test gate

Keep `OPEN_FINAL_TEST=False` while reading the validation result. The H6 thresholds, perturbation rules, consistency weight, and selected checkpoints must not be changed after seeing validation.

After this notebook is complete and the Model 1–6 protocol is frozen, set `OPEN_FINAL_TEST=True` and run only the final cell in the same Kaggle session. It evaluates both the matched control and Model 6 on clean test text and on the same three deterministic perturbation variants. The clean test remains the paper's main benchmark; perturbed test results are the robustness benchmark.


In [ ]:
# This block remains inactive during model development.
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    test_variants = [
        build_variant_frame(test_df, ROBUSTNESS_SEED, variant_id)
        for variant_id in range(N_VALIDATION_VARIANTS)
    ]
    final_rows = []
    for arm_name in ARM_CONFIGS:
        for run_seed in RUN_SEEDS:
            state_path = MODEL_DIR / arm_name / f'seed_{run_seed}' / 'best_state.pt'
            if not state_path.exists():
                raise FileNotFoundError(f'Missing state: {state_path}')
            model = build_fresh_model()
            model.load_state_dict(load_exact_state(state_path), strict=True)
            metrics, _ = evaluate_clean_and_variants(
                model, test_df, test_variants,
                MODEL_DIR / arm_name / f'seed_{run_seed}' / 'FINAL_test',
                split_name='FINAL_test',
            )
            final_rows.append({'arm': arm_name, 'seed': run_seed, **metrics})
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    final_results = pd.DataFrame(final_rows).sort_values(['arm', 'seed'])
    final_summary = final_results.groupby('arm')[summary_metrics].agg(['mean', 'std'])
    final_results.to_csv(MODEL_DIR / 'FINAL_test_seed_results.csv', index=False)
    final_summary.to_csv(MODEL_DIR / 'FINAL_test_mean_std.csv')
    display(final_results.round(4))
    display(final_summary.round(4))
